# 04 — Pull Conflict Event Labels

This notebook pulls the three datasets that provide **ground-truth labels** for the
civil war, coup, and state failure outcome models.

| Dataset | Label(s) produced | Access |
|---|---|---|
| **UCDP-GED v24** | `civil_war_onset`, `civil_war_incidence`, battle deaths | REST API (ucdpapi.pcr.uu.se) |
| **Powell-Thyne Coup Dataset** | `coup_attempt`, `coup_success` | Direct CSV download |
| **PITF State Failure Problem Set** | `ethwar`, `revwar`, `genocide`, `adverse`, `sftprev` | Direct CSV download |

## What this notebook does
1. Downloads UCDP-GED event data via the UCDP REST API (paginated)
2. Downloads Powell-Thyne coup list
3. Downloads PITF State Failure Problem Set
4. Writes each to ADLS as raw parquet
5. Constructs a combined **country-month label table** for model training

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import time
import requests
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# UCDP GED REST API
UCDP_GED_VERSION  = "24.1"   # update when UCDP releases a new version
UCDP_API_BASE     = f"https://ucdpapi.pcr.uu.se/api/gedevents/{UCDP_GED_VERSION}"
UCDP_PAGE_SIZE    = 1000

# Powell-Thyne coup dataset (Thyne & Powell, UCF — updated annually)
POWELL_THYNE_URL  = (
    "https://www.uky.edu/~clthyn2/coup_data/powell_thyne_coups_final.txt"
)

# PITF State Failure Problem Set (Center for Systemic Peace)
PITF_URL = "https://www.systemicpeace.org/inscr/PITF%20SPSS%20Data%202022.csv"

print(f"Run date    : {RUN_DATE}")
print(f"UCDP GED    : v{UCDP_GED_VERSION}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## 1. UCDP-GED (Uppsala Conflict Data Program — Georeferenced Event Dataset)

**Provider:** UCDP, Uppsala University  
**Coverage:** Global, 1989–present  
**Frequency:** Annual release (candidate events updated monthly)  
**Unit of observation:** A conflict incident — a reported use of armed force by an organised
actor resulting in ≥1 battle death (state-based / non-state) or ≥25 civilian deaths
(one-sided violence).

We pull all three violence types:
- `type_of_violence=1` — state-based (≥1 actor is a government; ≥25 battle deaths/year threshold)
- `type_of_violence=2` — non-state conflict (neither actor is a government)
- `type_of_violence=3` — one-sided violence (organised actor uses force against civilians)

### Key fields (full schema)

| Field | Type | Description |
|---|---|---|
| `id` | integer | Unique event ID |
| `year` | integer | Year of event |
| `type_of_violence` | integer | 1=State-based, 2=Non-state, 3=One-sided violence |
| `conflict_new_id` | integer | Conflict ID linking events to the same conflict |
| `dyad_new_id` | integer | Dyad (pair of actors) identifier |
| `side_a` | string | Government/state side actor name |
| `side_b` | string | Opposing actor name |
| `country` | string | Country where event occurred |
| `country_id` | integer | UCDP country identifier |
| `region` | string | Geographic region |
| `date_start` | date | Start date of event |
| `date_end` | date | End date of event |
| `deaths_a` | integer | Best estimate of deaths on Side A |
| `deaths_b` | integer | Best estimate of deaths on Side B |
| `deaths_civilians` | integer | Best estimate of civilian deaths |
| `deaths_unknown` | integer | Deaths with unknown affiliation |
| `best` | integer | Best estimate of total deaths |
| `high` | integer | High-end estimate of total deaths |
| `low` | integer | Low-end estimate of total deaths |
| `latitude` | float | Latitude of event location |
| `longitude` | float | Longitude of event location |
| `geom_wkt` | string | Event location as WKT polygon/point |
| `priogrid_gid` | integer | PRIO-GRID cell ID — joins to notebook 09 (PRIO-GRID) |
| `source_article` | string | Source article citation |

In [ ]:
def fetch_ucdp_page(page: int, pagesize: int = UCDP_PAGE_SIZE) -> dict:
    params = {"pagesize": pagesize, "page": page}
    resp = requests.get(UCDP_API_BASE, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

def pull_all_ucdp_ged() -> pd.DataFrame:
    all_records = []
    page = 1
    # First call to get total count
    first = fetch_ucdp_page(page=1, pagesize=1)
    total = first.get("TotalCount", 0)
    total_pages = (total // UCDP_PAGE_SIZE) + 1
    print(f"UCDP-GED: {total:,} events across ~{total_pages} pages")

    pbar = tqdm(total=total_pages, desc="UCDP pages", unit="page")
    while True:
        data = fetch_ucdp_page(page)
        records = data.get("Result", [])
        if not records:
            break
        all_records.extend(records)
        pbar.update(1)
        pbar.set_postfix({"total": len(all_records)})
        if len(records) < UCDP_PAGE_SIZE:
            break
        page += 1
        time.sleep(0.2)  # be polite to the UCDP API
    pbar.close()
    return pd.DataFrame(all_records)

print("Pulling UCDP-GED...")
df_ucdp = pull_all_ucdp_ged()
print(f"Total UCDP-GED events: {len(df_ucdp):,}")
df_ucdp.head(2)

In [ ]:
# --- Documented schema ---
# All 24 key fields from UCDP-GED (DatasetVariableSynthesis.md §1.1)
UCDP_INT_COLS = [
    "id",                # Unique event ID
    "year",              # Year of event
    "type_of_violence",  # 1=State-based, 2=Non-state, 3=One-sided
    "conflict_new_id",   # Conflict ID (links events in the same conflict)
    "dyad_new_id",       # Dyad identifier (pair of actors)
    "country_id",        # UCDP country identifier
    "deaths_a",          # Deaths on Side A
    "deaths_b",          # Deaths on Side B
    "deaths_civilians",  # Civilian deaths
    "deaths_unknown",    # Deaths of unknown affiliation
    "best",              # Best estimate of total deaths
    "high",              # High-end estimate
    "low",               # Low-end estimate
    "priogrid_gid",      # PRIO-GRID cell ID (spatial join key with notebook 09)
]

UCDP_FLOAT_COLS = [
    "latitude",          # Latitude of event location
    "longitude",         # Longitude of event location
]

UCDP_DATE_COLS = [
    "date_start",        # Start date of event
    "date_end",          # End date of event
]

# String columns kept as-is: side_a, side_b, country, region, geom_wkt, source_article

# --- Apply casts ---
for col in UCDP_INT_COLS:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_numeric(df_ucdp[col], errors="coerce").astype("Int64")

for col in UCDP_FLOAT_COLS:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_numeric(df_ucdp[col], errors="coerce")

for col in UCDP_DATE_COLS:
    if col in df_ucdp.columns:
        df_ucdp[col] = pd.to_datetime(df_ucdp[col], errors="coerce")

# Derive year_month for monthly panel joins
if "date_start" in df_ucdp.columns:
    df_ucdp["year_month"] = df_ucdp["date_start"].dt.to_period("M").astype(str)

# --- Schema validation ---
all_expected = UCDP_INT_COLS + UCDP_FLOAT_COLS + UCDP_DATE_COLS + [
    "side_a", "side_b", "country", "region", "geom_wkt", "source_article",
]
missing_fields = [f for f in all_expected if f not in df_ucdp.columns]
if missing_fields:
    print(f"Note: {len(missing_fields)} schema field(s) absent from this API version: {missing_fields}")
else:
    print("All 24 documented schema fields present.")

# --- Summary statistics ---
print(f"\nYears        : {df_ucdp['year'].min()}–{df_ucdp['year'].max()}")
print(f"Countries    : {df_ucdp['country'].nunique() if 'country' in df_ucdp.columns else 'n/a'}")
print(f"Conflicts    : {df_ucdp['conflict_new_id'].nunique() if 'conflict_new_id' in df_ucdp.columns else 'n/a'} unique conflict IDs")
print(f"Dyads        : {df_ucdp['dyad_new_id'].nunique() if 'dyad_new_id' in df_ucdp.columns else 'n/a'} unique dyads")
print(f"Total deaths : {df_ucdp['best'].sum():,.0f} (best estimate)")
print()
vt_labels = {1: "state-based", 2: "non-state", 3: "one-sided"}
vc = df_ucdp["type_of_violence"].value_counts().sort_index()
for code, count in vc.items():
    print(f"  type_of_violence={code} ({vt_labels.get(int(code), '?')}): {count:,} events")

### UCDP-GED: country-year death aggregates

In addition to the raw event file, we build a **country-year** summary that is used
directly as the `civil_war_incidence` predictor feature and the `battle_deaths` regression
target in the panel model. The `conflict_new_id` column is preserved so that
`build_panel.py` can apply the ≥24-month peace-spell filter when constructing
the `civil_war_onset` binary label.

In [ ]:
if "country" in df_ucdp.columns and "year" in df_ucdp.columns:
    # State-based events only (type_of_violence==1) for civil-war labels
    df_state = df_ucdp[df_ucdp["type_of_violence"] == 1].copy()

    death_cols = {c: "sum" for c in ["best", "high", "low", "deaths_civilians"] if c in df_state.columns}
    death_cols["conflict_new_id"] = "nunique"   # distinct active conflicts in country-year

    df_ucdp_cy = (
        df_state
        .groupby(["country", "country_id", "year"], as_index=False)
        .agg(death_cols)
        .rename(columns={
            "best":             "battle_deaths_best",
            "high":             "battle_deaths_high",
            "low":              "battle_deaths_low",
            "deaths_civilians": "civilian_deaths",
            "conflict_new_id":  "n_active_conflicts",
        })
    )

    # Binary incidence flag: any state-based conflict active this country-year
    df_ucdp_cy["civil_war_incidence"] = (df_ucdp_cy["n_active_conflicts"] > 0).astype("Int8")

    print(f"Country-year aggregate: {df_ucdp_cy.shape}")
    print(f"Country-years with active conflict: {df_ucdp_cy['civil_war_incidence'].sum():,}")
    df_ucdp_cy.head(3)
else:
    print("Skipping country-year aggregate — 'country' or 'year' column missing")
    df_ucdp_cy = pd.DataFrame()

In [ ]:
write_parquet(df_ucdp,    f"raw/ucdp_ged/{RUN_DATE}/ucdp_ged_events.parquet")
if not df_ucdp_cy.empty:
    write_parquet(df_ucdp_cy, f"raw/ucdp_ged/{RUN_DATE}/ucdp_ged_country_year.parquet")

## 2. Powell-Thyne Coup Dataset

In [ ]:
print(f"Downloading Powell-Thyne from {POWELL_THYNE_URL} ...")
resp = requests.get(POWELL_THYNE_URL, timeout=60)
resp.raise_for_status()

# File is tab-delimited
df_coups = pd.read_csv(io.BytesIO(resp.content), sep="\t", low_memory=False)
print(f"Raw shape: {df_coups.shape}")
print(f"Columns: {list(df_coups.columns)}")
df_coups.head(3)

In [ ]:
# Standardise column names (Powell-Thyne schema varies slightly by release year)
col_map = {
    c: c.lower().strip().replace(" ", "_") for c in df_coups.columns
}
df_coups.rename(columns=col_map, inplace=True)

for col in ["year", "month", "day", "coup"]:
    if col in df_coups.columns:
        df_coups[col] = pd.to_numeric(df_coups[col], errors="coerce").astype("Int64")

# coup == 1 → successful; coup == 2 → attempted/failed
if "coup" in df_coups.columns:
    df_coups["coup_success"]  = (df_coups["coup"] == 1).astype("Int8")
    df_coups["coup_attempt"]  = df_coups["coup"].isin([1, 2]).astype("Int8")

print(f"Coups: {len(df_coups):,} events | {df_coups['year'].min()}–{df_coups['year'].max()}")
print(df_coups["coup"].value_counts().rename({1: 'successful', 2: 'attempted'}))

In [ ]:
write_parquet(df_coups, f"raw/powell_thyne/{RUN_DATE}/powell_thyne_coups.parquet")

## 3. PITF State Failure Problem Set

In [ ]:
print(f"Downloading PITF State Failure data...")
resp = requests.get(PITF_URL, timeout=60)
resp.raise_for_status()

df_pitf = pd.read_csv(io.BytesIO(resp.content), low_memory=False)
print(f"Raw shape: {df_pitf.shape}")
df_pitf.head(3)

In [ ]:
# Standardise column names
df_pitf.columns = [c.lower().strip() for c in df_pitf.columns]

# Key binary onset labels
label_cols = ["sftprev", "ethwar", "revwar", "genocide", "adverse"]
for col in label_cols:
    if col in df_pitf.columns:
        df_pitf[col] = pd.to_numeric(df_pitf[col], errors="coerce").astype("Int8")

if "year" in df_pitf.columns:
    df_pitf["year"] = pd.to_numeric(df_pitf["year"], errors="coerce").astype("Int64")

present_labels = [c for c in label_cols if c in df_pitf.columns]
print(f"PITF shape: {df_pitf.shape}")
print(f"Label columns found: {present_labels}")
if present_labels:
    print(df_pitf[present_labels].sum())

In [ ]:
write_parquet(df_pitf, f"raw/pitf/{RUN_DATE}/pitf_state_failure.parquet")

## Summary

In [ ]:
print("=" * 55)
print("Conflict labels pull complete")
print("=" * 55)
print(f"  UCDP-GED events    : {len(df_ucdp):,} rows | {df_ucdp['year'].min()}–{df_ucdp['year'].max()}")
if not df_ucdp_cy.empty:
    print(f"  UCDP-GED cty-year  : {len(df_ucdp_cy):,} country-years "
          f"| {int(df_ucdp_cy['civil_war_incidence'].sum())} with active conflict")
print(f"  Powell-Thyne       : {len(df_coups):,} coup events")
print(f"  PITF               : {len(df_pitf):,} country-years")
print()
print("ADLS paths written:")
print(f"  raw/ucdp_ged/{RUN_DATE}/ucdp_ged_events.parquet")
if not df_ucdp_cy.empty:
    print(f"  raw/ucdp_ged/{RUN_DATE}/ucdp_ged_country_year.parquet")
print(f"  raw/powell_thyne/{RUN_DATE}/powell_thyne_coups.parquet")
print(f"  raw/pitf/{RUN_DATE}/pitf_state_failure.parquet")